# JWT Validation

This notebook covers:

1. What a JWT actually is on the wire — header, payload, signature
2. Encoding a token with `python-jose` and HS256
3. Decoding and verifying a token: signature first, then claims
4. Claim validation: `exp` (expiry), `iss` (issuer), `aud` (audience)
5. Propagating identity from the decoded payload through `Depends(get_current_user)`
6. Mapping every JWT failure mode to a clean **401**
7. Key rotation: `kid` headers and verification keysets

**Scope**: FastAPI + `python-jose[cryptography]` + `TestClient`. This notebook replaces the opaque-token lookup from 5.1 with a self-describing token; everything else (the `/token` endpoint shape, `get_current_user`, the 401-vs-403 split) stays the same.

## 1. JWT Anatomy: Header, Payload, Signature

A JWT (RFC 7519) is three base64url-encoded blobs joined by dots:

```
<header>.<payload>.<signature>
```

- **Header** — JSON describing how the token is signed: `{"alg": "HS256", "typ": "JWT"}`. The `alg` tells the verifier which algorithm to expect; the `kid` (key id) tells it which key.
- **Payload** — JSON of *claims*. Standard claim names (RFC 7519): `iss` (issuer), `sub` (subject — usually the user id), `aud` (audience — who the token is for), `exp` (expiry, seconds since epoch), `iat` (issued at), `nbf` (not before). You can add app-specific claims (`roles`, `org_id`, ...) — keep them small; the token rides in every request.
- **Signature** — `HMAC_SHA256(secret, base64(header) + "." + base64(payload))` for HS256, or an RSA signature for RS256. The signature is what makes the token *unforgeable* — a tamper to the payload invalidates the signature.

The headline property: the verifier only needs the **secret** (HS256) or the **public key** (RS256) to validate. **No database lookup per request.** That's the structural advantage over the opaque-token lookup from notebook 5.1 — JWTs are *stateless* on the server side. The cost is that you can't revoke a single token instantly; you wait for `exp`, or rotate the signing key.

Two failure modes are non-negotiable: a token with a bad signature must be rejected (or the whole scheme is meaningless), and an expired token must be rejected (or stolen tokens live forever). Both produce **401**.

## 2. Encoding a Token

`python-jose` exposes `jwt.encode(payload, key, algorithm)` and `jwt.decode(token, key, algorithms=[...])`. Build the payload as a plain dict — `python-jose` does the JSON serialization and base64url encoding for you.

Some discipline up front:

- **`exp` is required.** Don't issue non-expiring tokens. 15 minutes is a reasonable access-token TTL; use refresh tokens (out of scope here) for longer sessions.
- **`iss` and `aud` are recommended** for any real system. They prevent a token issued for service A from being accepted by service B.
- **Times are integer Unix seconds**, not `datetime` objects — convert with `int(datetime.now(tz=timezone.utc).timestamp())`.
- **Never put secrets in the payload.** It's base64-encoded, *not* encrypted. Anyone with the token can read every claim.

In [ ]:
import base64, json
from datetime import datetime, timedelta, timezone
from jose import jwt

# In production this comes from settings (notebook 4.3) and is rotated periodically.
SECRET_KEY = "learn-jwt-secret-do-not-use-in-prod"
ALGORITHM  = "HS256"
ISSUER     = "portfolio-api"
AUDIENCE   = "portfolio-clients"

now = datetime.now(tz=timezone.utc)
claims = {
    "sub":   "alice",                                  # the user id
    "iss":   ISSUER,
    "aud":   AUDIENCE,
    "iat":   int(now.timestamp()),
    "exp":   int((now + timedelta(minutes=15)).timestamp()),
    "roles": ["trader"],                                # app-specific claim
}
token = jwt.encode(claims, SECRET_KEY, algorithm=ALGORITHM)
print("length:", len(token))
print("token :", token[:60], "...")

# Peel back the layers: the payload is plain base64url JSON. Anyone with the token can read it.
def b64url_decode(segment: str) -> bytes:
    pad = "=" * (-len(segment) % 4)
    return base64.urlsafe_b64decode(segment + pad)

header_b64, payload_b64, signature_b64 = token.split(".")
print("\nheader  :", json.loads(b64url_decode(header_b64)))
print("payload :", json.loads(b64url_decode(payload_b64)))
print("sig len :", len(b64url_decode(signature_b64)), "bytes (HMAC-SHA256 is 32)")

Three observations that drive every later section:

- The **payload is plaintext** to anyone holding the token. Treat it as public.
- The **signature is the only thing that ties the payload to the issuer.** Without it (or with the wrong key) the payload is unverifiable.
- The **header advertises the algorithm**. That's a known footgun — the historical `alg="none"` attack relied on verifiers trusting the header. `python-jose` mitigates this by requiring you to pass an `algorithms=` *whitelist* when decoding. We always do.

## 3. Decoding and Verifying

`jwt.decode(token, key, algorithms=[...], audience=..., issuer=...)` runs every check the spec requires:

1. Parses the three segments and the header.
2. Confirms `header.alg` is in your `algorithms` list (kills the `alg=none` attack).
3. Verifies the signature using `key`.
4. Verifies `exp`, `nbf`, `iat` against the current time.
5. Verifies `aud` and `iss` if you pass them.

If *anything* fails, it raises a `JWTError` subclass: `ExpiredSignatureError`, `JWTClaimsError`, or the base `JWTError`. The route layer turns all of them into 401.

In [ ]:
from jose import JWTError

# Happy path: same secret, declared algorithm, audience and issuer match.
decoded = jwt.decode(
    token,
    SECRET_KEY,
    algorithms=[ALGORITHM],
    audience=AUDIENCE,
    issuer=ISSUER,
)
print("decoded sub:  ", decoded["sub"])
print("decoded roles:", decoded["roles"])

# Wrong secret -> signature verification fails.
try:
    jwt.decode(token, "wrong-secret", algorithms=[ALGORITHM], audience=AUDIENCE, issuer=ISSUER)
except JWTError as e:
    print("\nwrong secret  ->", type(e).__name__, "|", e)

# Missing the algorithms whitelist would be a CVE waiting to happen.
# python-jose makes this a hard error rather than silently trusting header.alg.
try:
    jwt.decode(token, SECRET_KEY, algorithms=["RS256"])  # token was HS256
except JWTError as e:
    print("algorithm mismatch ->", type(e).__name__, "|", e)

Read the failure messages carefully — they're the exact strings you'll be tempted to put in your 401 body. **Don't.** A response like `"Signature verification failed"` invites probing. Return a generic `"Invalid token"` to the client and keep the structured error in your *logs* (notebook 6.2).

The `algorithms=` list is the single most important security knob in the whole `jwt.decode` call. Always pass it. Always pin it to the exact algorithm the issuer uses. *Never* let the header drive the choice.

In [ ]:
from jose.exceptions import ExpiredSignatureError, JWTClaimsError

def issue(extra: dict, ttl: timedelta = timedelta(minutes=15)) -> str:
    now = datetime.now(tz=timezone.utc)
    payload = {
        "sub": "alice",
        "iss": ISSUER,
        "aud": AUDIENCE,
        "iat": int(now.timestamp()),
        "exp": int((now + ttl).timestamp()),
        **extra,
    }
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)

# 1) Already-expired token.
stale = issue({}, ttl=timedelta(seconds=-1))
try:
    jwt.decode(stale, SECRET_KEY, algorithms=[ALGORITHM], audience=AUDIENCE, issuer=ISSUER)
except ExpiredSignatureError as e:
    print("expired ->", type(e).__name__, "|", e)

# 2) Wrong audience: token was issued for portfolio-clients but the verifier expects analytics-clients.
good = issue({})
try:
    jwt.decode(good, SECRET_KEY, algorithms=[ALGORITHM], audience="analytics-clients", issuer=ISSUER)
except JWTClaimsError as e:
    print("bad audience ->", type(e).__name__, "|", e)

# 3) Wrong issuer: a token from a different service can't pass for ours.
try:
    jwt.decode(good, SECRET_KEY, algorithms=[ALGORITHM], audience=AUDIENCE, issuer="some-other-service")
except JWTClaimsError as e:
    print("bad issuer   ->", type(e).__name__, "|", e)

# 4) Tampered payload: flip one byte in the base64 segment and the signature breaks.
header_b64, payload_b64, sig_b64 = good.split(".")
tampered = f"{header_b64}.{payload_b64[:-1]}{'A' if payload_b64[-1] != 'A' else 'B'}.{sig_b64}"
try:
    jwt.decode(tampered, SECRET_KEY, algorithms=[ALGORITHM], audience=AUDIENCE, issuer=ISSUER)
except JWTError as e:
    print("tampered     ->", type(e).__name__, "|", e)

Four classes of failure, all mapped to one place at the route layer:

| Cause                         | Exception class             | Status to return |
|------------------------------ |---------------------------- |------------------|
| Past `exp`                    | `ExpiredSignatureError`     | 401              |
| `aud` / `iss` mismatch         | `JWTClaimsError`            | 401              |
| Bad signature / tampered payload | `JWTError` (base)         | 401              |
| Malformed (not three segments)| `JWTError` (base)           | 401              |

Authorization decisions ("this token is valid but lacks the `trader` role") stay at 403 — those happen *after* decode succeeds, as in 5.1.

## 5. Carrying Identity via `Depends`

Now the FastAPI integration. The `get_current_user` shape from 5.1 stays — what changes is the verification step. Instead of looking the token up in a `TOKENS` dict, we *decode and verify* it. No server-side state.

The flow:

- `oauth2_scheme` parses the `Authorization` header → raw token string.
- `decode_token(token)` runs `jwt.decode` with the full whitelist, maps every JWT exception to a 401.
- `get_current_user` reads `sub` from the verified payload, looks up the user in the user store (still needed — `sub` is just an id), and returns the full `UserInDB`.
- `require_role(...)` chains on top, exactly as in 5.1.

Notice the user store **stays**. The token tells us *who* is making the request; the user store tells us whether that user is active, what roles they have, etc. The roles on the token are a *cache* — if you put them in the JWT for convenience, remember they go stale until the next token issuance.

In [ ]:
import secrets
from fastapi import Depends, FastAPI, HTTPException, status
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from fastapi.testclient import TestClient
from passlib.context import CryptContext
from pydantic import BaseModel

pwd_context = CryptContext(schemes=["pbkdf2_sha256"], deprecated="auto")

class UserInDB(BaseModel):
    username: str
    hashed_password: str
    disabled: bool = False
    roles: list[str] = []

USERS: dict[str, UserInDB] = {
    "alice": UserInDB(username="alice", hashed_password=pwd_context.hash("alice-pw"), roles=["trader"]),
    "bob":   UserInDB(username="bob",   hashed_password=pwd_context.hash("bob-pw"),   roles=["viewer"]),
}

app = FastAPI()
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/token")

def create_access_token(*, sub: str, roles: list[str], ttl: timedelta = timedelta(minutes=15)) -> str:
    now = datetime.now(tz=timezone.utc)
    payload = {
        "sub":   sub,
        "iss":   ISSUER,
        "aud":   AUDIENCE,
        "iat":   int(now.timestamp()),
        "exp":   int((now + ttl).timestamp()),
        "roles": roles,
    }
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)

def decode_token(token: str) -> dict:
    try:
        return jwt.decode(
            token,
            SECRET_KEY,
            algorithms=[ALGORITHM],
            audience=AUDIENCE,
            issuer=ISSUER,
        )
    except JWTError:
        # Collapse every JWT failure (expired, bad sig, bad aud/iss, malformed) into one 401.
        # The structured cause goes in the logs, not the response.
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid token",
            headers={"WWW-Authenticate": "Bearer"},
        )

def get_current_user(token: str = Depends(oauth2_scheme)) -> UserInDB:
    payload = decode_token(token)
    username = payload.get("sub")
    user = USERS.get(username) if username else None
    if user is None or user.disabled:
        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED, detail="User unavailable")
    return user

def require_role(role: str):
    def dep(user: UserInDB = Depends(get_current_user)) -> UserInDB:
        if role not in user.roles:
            raise HTTPException(
                status_code=status.HTTP_403_FORBIDDEN,
                detail=f"Requires role '{role}'",
            )
        return user
    return dep

class TokenResponse(BaseModel):
    access_token: str
    token_type: str = "bearer"

@app.post("/token", response_model=TokenResponse)
def login(form: OAuth2PasswordRequestForm = Depends()):
    user = USERS.get(form.username)
    if user is None or not pwd_context.verify(form.password, user.hashed_password):
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Incorrect username or password",
            headers={"WWW-Authenticate": "Bearer"},
        )
    return TokenResponse(access_token=create_access_token(sub=user.username, roles=user.roles))

@app.get("/portfolios/me")
def my_portfolio(user: UserInDB = Depends(get_current_user)):
    return {"user": user.username, "roles": user.roles}

@app.post("/portfolios/me/trades")
def place_trade(user: UserInDB = Depends(require_role("trader"))):
    return {"placed_by": user.username, "status": "accepted"}

client = TestClient(app)

alice_token = client.post("/token", data={"username": "alice", "password": "alice-pw"}).json()["access_token"]
bob_token   = client.post("/token", data={"username": "bob",   "password": "bob-pw"}).json()["access_token"]
print("alice token (truncated):", alice_token[:40], "...")

print("\nalice /me              :", client.get("/portfolios/me", headers={"Authorization": f"Bearer {alice_token}"}).status_code)
print("alice /trades (trader) :", client.post("/portfolios/me/trades", headers={"Authorization": f"Bearer {alice_token}"}).status_code)
print("bob   /trades (viewer) :", client.post("/portfolios/me/trades", headers={"Authorization": f"Bearer {bob_token}"}).status_code)

Compare the routes to the ones in 5.1. They're identical. That's the whole point: **swapping the verification strategy doesn't touch the route layer.** A capstone app can start with opaque tokens, switch to JWTs in a single PR, and not change a single route signature.

## 6. Handling Tampered or Expired Tokens

Let's drive the four failure modes through the actual endpoint and confirm they all surface as 401. The route layer doesn't care *why* the token was bad — only the logs do.

In [ ]:
# 1) Expired token: issue one with a negative TTL.
expired = create_access_token(sub="alice", roles=["trader"], ttl=timedelta(seconds=-1))
r = client.get("/portfolios/me", headers={"Authorization": f"Bearer {expired}"})
print("expired         :", r.status_code, r.json())

# 2) Token signed with the wrong secret (e.g., issued by a compromised or rotated key).
wrong_secret_token = jwt.encode(
    {"sub": "alice", "iss": ISSUER, "aud": AUDIENCE,
     "iat": int(datetime.now(tz=timezone.utc).timestamp()),
     "exp": int((datetime.now(tz=timezone.utc) + timedelta(minutes=5)).timestamp()),
     "roles": ["trader"]},
    "some-other-secret",
    algorithm=ALGORITHM,
)
r = client.get("/portfolios/me", headers={"Authorization": f"Bearer {wrong_secret_token}"})
print("wrong signature :", r.status_code, r.json())

# 3) Tampered payload — flip a byte after issuance.
h, p, s = alice_token.split(".")
tampered = f"{h}.{p[:-1]}{'A' if p[-1] != 'A' else 'B'}.{s}"
r = client.get("/portfolios/me", headers={"Authorization": f"Bearer {tampered}"})
print("tampered payload:", r.status_code, r.json())

# 4) Token claims a non-existent user. Signature is valid; user lookup fails.
phantom = create_access_token(sub="mallory", roles=["trader"])
r = client.get("/portfolios/me", headers={"Authorization": f"Bearer {phantom}"})
print("unknown user    :", r.status_code, r.json())

# 5) Authenticated but missing role -> 403, NOT 401.
r = client.post("/portfolios/me/trades", headers={"Authorization": f"Bearer {bob_token}"})
print("bob lacks role  :", r.status_code, r.json())

Four ways to fail authentication, one status code: **401**, with `detail: "Invalid token"` (or `"User unavailable"` when the cryptographic check passes but the user is gone). Failure of *authorization* — case 5 — stays at 403, same as 5.1.

Resist the urge to leak which check failed in the response body. If a client wants to know whether their token expired, they can decode the unsigned payload locally (it's just base64) and check `exp` themselves. We keep the diagnostic precision *in our logs*.

## 7. Key Rotation: `kid` and Keysets

JWT secrets rot. You change them when:

- A secret might have leaked.
- Someone with access to it leaves the team.
- It's been a while ("a while" depends on your threat model — quarterly is reasonable).

If you simply replace the secret and redeploy, every outstanding token immediately becomes a 401 and every user has to log in again. The standard fix is **key rotation with overlap**:

1. Issue tokens with a `kid` header naming the current signing key.
2. The verifier keeps a *keyset* — both the new key and the previous key.
3. The verifier picks the key by `kid` and validates.
4. After all old tokens expire (TTL passes), drop the old key.

This is exactly how the production JWT story works (Auth0, Cognito, internal IdP — all publish a JWKS endpoint with multiple keys keyed by `kid`). HS256 with a single secret is fine for a tutorial; production-grade rotation usually means **RS256**, where each verifier holds the public keyset and only the issuer holds the private key. Here's the `kid` mechanics in miniature:

In [ ]:
# A keyset: kid -> secret. In production this is fetched from a JWKS endpoint at startup,
# refreshed periodically, and contains public keys for RS256.
KEYSET = {
    "key-2026-q2": "new-secret-after-rotation",
    "key-2026-q1": SECRET_KEY,            # still valid for tokens issued before rotation
}
CURRENT_KID = "key-2026-q2"

def issue_with_kid(sub: str, kid: str, ttl: timedelta = timedelta(minutes=15)) -> str:
    now = datetime.now(tz=timezone.utc)
    return jwt.encode(
        {"sub": sub, "iss": ISSUER, "aud": AUDIENCE,
         "iat": int(now.timestamp()),
         "exp": int((now + ttl).timestamp())},
        KEYSET[kid],
        algorithm=ALGORITHM,
        headers={"kid": kid},   # <-- the only difference from before
    )

def decode_with_keyset(token: str) -> dict:
    header = jwt.get_unverified_header(token)   # safe: doesn't trust signature
    kid = header.get("kid")
    key = KEYSET.get(kid)
    if key is None:
        raise HTTPException(status_code=401, detail="Unknown key id")
    return jwt.decode(token, key, algorithms=[ALGORITHM], audience=AUDIENCE, issuer=ISSUER)

# Token issued with the OLD key (before rotation): still validates.
old_token = issue_with_kid("alice", kid="key-2026-q1")
# Token issued with the NEW key (after rotation): also validates.
new_token = issue_with_kid("alice", kid="key-2026-q2")

print("old kid token verifies:", decode_with_keyset(old_token)["sub"])
print("new kid token verifies:", decode_with_keyset(new_token)["sub"])

# Once we drop the old key from the keyset, the old token stops validating — but by then,
# every outstanding token from before rotation has expired (TTL passed) and no client is
# still holding one. That's the overlap window.
del KEYSET["key-2026-q1"]
try:
    decode_with_keyset(old_token)
except HTTPException as e:
    print("after retirement:", e.status_code, e.detail)

Two cautions on the `kid` lookup:

- **Treat `kid` as untrusted input.** It's a header attribute the client controls. Look it up in *your* keyset — if it's not there, reject. Never construct a key path or filesystem path from it.
- **Keep the keyset small** (current + previous, typically). Rotation overlap shouldn't last longer than the token TTL plus a small buffer.

For HS256 you protect the secret like a database password. For RS256 the *private* key lives only on the issuer; verifiers hold only the public key, which they can fetch from a JWKS endpoint — that's how you decouple verification from issuance entirely. Same `kid` mechanic, different cryptography.

## Key Takeaways

- **A JWT is `header.payload.signature`** — two base64 JSON blobs and an HMAC (or RSA) signature. The payload is *readable*; the signature is what makes it *unforgeable*.
- **Stateless verification.** No database lookup per request. The cost is no instant revocation — you wait out `exp` or rotate keys.
- **Always pass `algorithms=[...]`** to `jwt.decode`. Never let the header pick. This closes the `alg=none` family of attacks.
- **Always set and verify `exp`**, plus `iss` and `aud` if you have any service boundaries. A token issued for service A must not be accepted by service B.
- **Collapse every JWT failure to 401** with a generic `"Invalid token"` body. The diagnostic detail goes in your logs, not in the response.
- **Identity and policy stay separate.** Decode produces the user; role/scope checks are layered on top and raise 403, not 401.
- **Rotate keys with overlap.** Tokens carry a `kid`; the verifier keeps a small keyset; drop old keys after the TTL expires.
- **Capstone tie-in**: `auth.py` will hold `create_access_token` + `decode_token` keyed by `settings.jwt_secret` / `settings.jwt_audience` (from 4.3). The route-layer `get_current_user` and `require_role` are unchanged from 5.1; only the verifier swaps out.

## Exercises

**1. Token-expiry test.** Issue a token with `ttl=timedelta(seconds=1)`. Hit `/portfolios/me` immediately — expect 200. `time.sleep(2)`. Hit it again — expect 401 with `detail="Invalid token"`. Write this as a single TestClient script and confirm both branches.

**2. Refresh tokens, briefly.** Add a `/refresh` endpoint that takes a long-lived refresh token (issue it from `/token` as a second field, `refresh_token`, signed with a *different* secret and `ttl=timedelta(days=7)`). `/refresh` accepts the refresh token, verifies it, and returns a fresh short-lived access token. Write a test that uses refresh after the access token expires. Sketch in a markdown cell *why* you'd put the refresh secret in a separate keyset (and rotate it independently).

**3. Per-tenant `aud` claims.** Suppose the portfolio API serves two tenants: `acme` and `globex`. Issue tokens with `aud="acme"` or `aud="globex"` from `/token` based on the user record. Make `decode_token` accept a `expected_aud` parameter and verify against it. Write a test where an `acme` token is rejected for a route mounted under `/globex/...`. This is the same pattern multi-tenant SaaS uses to keep tokens from leaking across customers.